# Minimum Viable EEG — sweep runner

Runs one validation shard, or the confirmatory test manifest, and writes one
result row per condition. Choose the phase with `SPLIT` in the configuration cell. See `docs/kaggle.md` for the full runbook.

Before you start:

1. **Accelerator → GPU T4 ×2** (Settings pane). T4 ×2 and P100 cost the same
   against your 30 h/week meter; T4 ×2 gives two GPUs per quota-hour.
2. **Environment → Original**, not Latest — `mne==1.8.0` breaks against newer
   scipy, and pinned dependencies are what make the sweep one experiment.
3. Attach both inputs: the cache dataset and the code dataset (or set
   `GITHUB_TOKEN` in Add-ons → Secrets).
4. The quota meter runs while a session is open **even when idle**. Stop it
   from Active Events when done — closing the tab is not enough.

Resumable: rows already in the output file are skipped, so a session that times
out costs nothing.

## 1 · Configuration — the only cell you normally edit

In [ ]:
SHARD_ID       = 0        # person i takes shard i
NUM_SHARDS     = 4
SPLIT          = "val"    # change to "test" for the confirmatory run only
KSTAR = None             # set to the validation-selected budget when SPLIT == "test"

# Kaggle mounts a dataset at /kaggle/input/datasets/<owner>/<slug>/ and keeps the
# top-level folder of the uploaded zip (cache/ and code/, see scripts/make_kaggle_bundle.py).
CACHE_DATASET  = "/kaggle/input/datasets/jackiewang2323/mve-eegmmidb-cache/cache"   # processed/S###/{X,y}.npy
CODE_DATASET   = "/kaggle/input/datasets/jackiewang2323/mve-code/code"
USE_GIT        = False
GIT_REPO       = "jaw039/min-viable-eeg"
GIT_COMMIT     = ""       # full SHA; never "latest main"

## 2 · Environment check — fail here, not three hours in

In [ ]:
import os, sys, json, subprocess, time
from pathlib import Path
import torch

print("torch      :", torch.__version__)
print("cuda avail :", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print("  gpu", i, ":", torch.cuda.get_device_name(i))
else:
    print("  !! No GPU. Settings -> Accelerator -> GPU T4 x2, then restart.")

assert Path(CACHE_DATASET).exists(), f"cache dataset not attached at {CACHE_DATASET}"

## 3 · Code, pinned to a commit

In [ ]:
REPO = Path("/kaggle/working/repo")

if USE_GIT:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if not REPO.exists():
        subprocess.run(["git", "clone",
                        f"https://{token}@github.com/{GIT_REPO}.git", str(REPO)], check=True)
    assert GIT_COMMIT, "pin GIT_COMMIT to a full SHA for reproducible runs"
    subprocess.run(["git", "-C", str(REPO), "checkout", GIT_COMMIT], check=True)
else:
    assert Path(CODE_DATASET).exists(), f"code dataset not attached at {CODE_DATASET}"
    if not REPO.exists():
        subprocess.run(["cp", "-r", CODE_DATASET, str(REPO)], check=True)

sys.path.insert(0, str(REPO))
os.chdir(REPO)

# Point the runner at the mounted cache instead of the absent local data dir.
os.environ["MVE_DATA_ROOT"] = str(Path(CACHE_DATASET) / "processed")

print("code at:", subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                                 capture_output=True, text=True).stdout.strip()
      or "(code dataset snapshot)")

## 4 · Frozen artifacts

Every result row is keyed to `artifacts/splits.json`, `artifacts/channel_ranking.json`
and `artifacts/budgets.json`. Check them before spending GPU time, not after.

In [ ]:
import hashlib
for name in ("splits.json", "channel_ranking.json", "budgets.json"):
    print(f"{name:<24} sha256:{hashlib.sha256(Path("artifacts", name).read_bytes()).hexdigest()[:16]}")

# The same drift guard the runner applies on every ranked run.
from src.channels import load_budgets, load_ranking, montage_order, select_channels
ch_names, ranked = montage_order(), load_ranking()[0]
for k_str, entry in load_budgets()["sets"].items():
    assert select_channels("ranked", int(k_str), ch_names, ranked) == entry["ranking_order"], \
        f"drift at k={k_str}"
print("\nranked selection matches budgets.json at every budget")

## 5 · Prepare the chosen phase

Validation runs a short smoke test before its shard. Test mode verifies the
complete validation report and prepares only k* and the 64-channel reference;
it skips the validation smoke test and validation shards. Set `SPLIT = "test"`
and `KSTAR` in the configuration cell for that phase.


In [ ]:
assert SPLIT in ("val", "test"), "SPLIT must be val or test"
if SPLIT == "test":
    if KSTAR is None:
        raise ValueError("Set KSTAR from the completed validation report before running test")
    subprocess.run([sys.executable, "scripts/run_sweep.py", "--write-test-manifest",
                    "--kstar-report", "results/kstar_report.json", "--kstar", str(KSTAR)], check=True)
    RUN_SHARD_ID, RUN_NUM_SHARDS = 0, 1
else:
    RUN_SHARD_ID, RUN_NUM_SHARDS = SHARD_ID, NUM_SHARDS
    subprocess.run([sys.executable, "scripts/run_sweep.py", "--smoke-test"], check=True)


In [ ]:
manifest = Path("manifests") / f"manifest_{SPLIT}.jsonl"
n_total = sum(1 for _ in manifest.open())
n_mine = len([i for i in range(n_total) if i % RUN_NUM_SHARDS == RUN_SHARD_ID])
print(f"manifest : {n_total} conditions")
print(f"my shard : {n_mine} conditions")
if SPLIT == "val":
    print("\nMultiply the smoke-test runtime above by that shard size.")
print("More than a few hours? Stop and re-plan before running.")

## 6 · Run the shard

Resumable and fault-tolerant: completed conditions are skipped, and a condition
that raises is recorded as an error row rather than killing the sweep.

In [ ]:
subprocess.run([sys.executable, "scripts/run_sweep.py", "--manifest", "--split", SPLIT,
                "--shard-id", str(RUN_SHARD_ID), "--num-shards", str(RUN_NUM_SHARDS)], check=True)


## 7 · Summary

`/kaggle/working` persists as notebook output, so this file can be attached
directly as an input to whoever merges the shards and runs `analyze.py`.

In [ ]:
from collections import defaultdict

out = Path("/kaggle/working/results") / f"shard_{RUN_SHARD_ID}_of_{RUN_NUM_SHARDS}_{SPLIT}.jsonl"
rows, errors = [], []
for line in out.open():
    r = json.loads(line)
    (errors if "error" in r else rows).append(r)

print(f"{len(rows)} results, {len(errors)} errors -> {out}\n")

by = defaultdict(list)
for r in rows:
    by[(r["budget_k"], r["selection"], r["training"])].append(r["kappa"])
print(f"{'k':>3}  {'selection':<14} {'training':<26} {'n':>3}  {'mean kappa':>10}")
for (k, sel, tr), ks in sorted(by.items()):
    print(f"{k:>3}  {sel:<14} {tr:<26} {len(ks):>3}  {sum(ks)/len(ks):>10.3f}")

for e in errors[:10]:
    print("ERROR:", e["error"])

## 8 · Save and audit the output

Download the saved Version's result file and audit it against the committed
manifest. The test phase has ten conditions when k* is below 64, or five when
k* is 64. It never runs another validation shard. Test performance remains
unreported until every planned test condition passes the audit.


In [ ]:
print("Completed phase:", SPLIT)
print("Result file:", out)
print("Download this saved Version's output, then run audit_results.py --require-complete")
